# Demo 3 — Exploratory to explanatory

Begin with an exploratory question and row/mark grain; before the final chart, name its audience, intended descriptive claim, displayed unit, and variable roles.

Colab is the default launch path; the same source runs in clean local Jupyter. The setup cell installs only mismatched course packages before their first import. Colab files are ephemeral, and edits made in a GitHub-opened Colab tab are not automatically saved to GitHub.

This demo uses only course-authored synthetic prepared data. Do not add uploads, Drive mounts, network data, credentials, private records, or sensitive output. Stored notebook output is not execution evidence: restart and run all from a fresh runtime. Assignment Colab submission remains conditional on the repository-save/Classroom50 pilot.


## Begin with bounded exploration

The exploratory question is how practice hours and observed scores vary across the ten prepared participant rows. Each point represents one participant. The view may reveal a pattern worth describing, but it does not establish correlation, uncertainty, prediction, or cause.


In [ ]:
import importlib.metadata
import platform
from pathlib import Path
import subprocess
import sys

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

assert platform.python_version() == "3.12.13", (
    "Select the course Python 3.12.13 runtime before continuing; found "
    f"{platform.python_version()}."
)

install_specs = []
for package_name, expected_version in EXPECTED_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        install_specs.append(f"{package_name}=={expected_version}")

if install_specs:
    print("Installing mismatched course packages:", ", ".join(install_specs))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *install_specs]
    )

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

actual_versions = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "Matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}
print(actual_versions)
assert actual_versions == {
    "Python": "3.12.13",
    "NumPy": "2.0.2",
    "pandas": "3.0.3",
    "Matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
PURPLE = "#CC79A7"


In [ ]:
from hashlib import sha256

FIXTURES = {'program_progress.csv': {'text': 'program,round_number,score\nStandard,1,62\nStandard,2,65\nStandard,3,67\nStandard,4,70\nStandard,5,72\nGuided,1,61\nGuided,2,66\nGuided,3,71\nGuided,4,75\nGuided,5,79\n', 'sha256': 'c48d53634f711d4f60b32f230633a47c77e56d8b1eac5f8c84fbad3858f85b36'}, 'participant_scores.csv': {'text': 'participant_id,program,practice_hours,score\nS01,Standard,2.0,61\nS02,Standard,3.0,65\nS03,Standard,3.5,66\nS04,Standard,4.0,67\nS05,Standard,5.0,69\nG01,Guided,2.5,64\nG02,Guided,3.5,70\nG03,Guided,4.0,73\nG04,Guided,5.0,77\nG05,Guided,6.0,82\n', 'sha256': '8eecd1393f3dbd4599269ba41724b28325ea2035f926ffeca674c2150abfc165'}}


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "07" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATHS = {}
for fixture_name, fixture_contract in FIXTURES.items():
    fixture_path = DATA_DIRECTORY / fixture_name
    if not fixture_path.exists():
        fixture_path.write_text(fixture_contract["text"], encoding="utf-8")
    actual_checksum = sha256(fixture_path.read_bytes()).hexdigest()
    assert actual_checksum == fixture_contract["sha256"], (
        f"{fixture_name} does not match the supplied fixture checksum. "
        "Restore the committed file; corrupt data are never replaced silently."
    )
    FIXTURE_PATHS[fixture_name] = fixture_path

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
for output_name in ['program_progress_explanatory.png', 'explanatory_supporting_data.csv', 'explanatory_text_alternative.txt']:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()

print("Demo directory:", DEMO_DIRECTORY)
print("Fixtures:", FIXTURE_PATHS)
print("Output directory:", OUTPUT_DIRECTORY)

progress = pd.read_csv(FIXTURE_PATHS["program_progress.csv"])
participants = pd.read_csv(FIXTURE_PATHS["participant_scores.csv"])
assert progress.shape == (10, 3)
assert participants.shape == (10, 4)
assert progress[["program", "round_number"]].duplicated().sum() == 0
assert participants["participant_id"].is_unique


In [ ]:
exploratory_figure, exploratory_ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(
    data=participants,
    x="practice_hours",
    y="score",
    hue="program",
    style="program",
    palette={"Standard": BLUE, "Guided": ORANGE},
    markers={"Standard": "o", "Guided": "s"},
    ax=exploratory_ax,
)
exploratory_ax.set(
    title="Exploratory view of practice and observed score",
    xlabel="Practice (hours)",
    ylabel="Observed score (points)",
)
exploratory_ax.legend(title="Program")
exploratory_figure.tight_layout()

exploratory_observation = (
    "In these ten prepared rows, larger practice-hour values appear with larger "
    "observed scores. This descriptive view does not establish why the values "
    "vary or whether the pattern generalizes."
)
assert exploratory_ax.get_xlabel() == "Practice (hours)"
assert exploratory_ax.get_ylabel() == "Observed score (points)"
assert "does not establish" in exploratory_observation
print(exploratory_observation)


## Focus one explanatory claim

- **Question:** How do observed prepared scores change across five rounds for two programs?
- **Audience:** A course coordinator deciding what result deserves follow-up.
- **Intended claim:** The Guided prepared score rises more and finishes seven points above Standard in round 5.
- **Unit and grain:** One point is one program-round prepared summary; one row stores that point.

The chart remains descriptive and does not establish a causal program effect.


In [ ]:
standard_rounds = progress.loc[progress["program"].eq("Standard")]
guided_rounds = progress.loc[progress["program"].eq("Guided")]

explanatory_figure, explanatory_ax = plt.subplots(figsize=(8, 4.8))
explanatory_ax.plot(
    standard_rounds["round_number"],
    standard_rounds["score"],
    color=BLUE,
    marker="o",
    linestyle="-",
    linewidth=2,
    label="Standard",
)
explanatory_ax.plot(
    guided_rounds["round_number"],
    guided_rounds["score"],
    color=ORANGE,
    marker="s",
    linestyle="--",
    linewidth=2,
    label="Guided",
)
annotation_text = "Round 5 observed separation: 7 points"
explanatory_annotation = explanatory_ax.annotate(
    annotation_text,
    xy=(5, 79),
    xytext=(3.05, 82),
    arrowprops={"arrowstyle": "->", "color": "#333333"},
)
explanatory_ax.set(
    title="Guided scores rise more in the prepared five-round summary",
    xlabel="Study round",
    ylabel="Prepared score (points)",
    xticks=[1, 2, 3, 4, 5],
    xlim=(0.8, 5.2),
    ylim=(58, 84),
)
explanatory_ax.legend(title="Program", frameon=False)
explanatory_ax.spines[["top", "right"]].set_visible(False)
explanatory_figure.tight_layout()

supporting_data = progress[["program", "round_number", "score"]].copy()
supporting_path = OUTPUT_DIRECTORY / "explanatory_supporting_data.csv"
supporting_data.to_csv(supporting_path, index=False)

explanatory_text_alternative = (
    "Line chart of prepared score by study round for Standard and Guided "
    "programs. Both rise across five rounds; Guided rises from 61 to 79 and "
    "finishes seven points above Standard. These prepared summaries are "
    "descriptive and do not establish a causal program effect."
)
text_path = OUTPUT_DIRECTORY / "explanatory_text_alternative.txt"
text_path.write_text(explanatory_text_alternative + "\n", encoding="utf-8")

explanatory_path = OUTPUT_DIRECTORY / "program_progress_explanatory.png"
explanatory_figure.savefig(explanatory_path, dpi=150, bbox_inches="tight")


In [ ]:
assert len(explanatory_ax.lines) == 2
assert [line.get_marker() for line in explanatory_ax.lines] == ["o", "s"]
assert [line.get_linestyle() for line in explanatory_ax.lines] == ["-", "--"]
assert explanatory_ax.get_xlabel() == "Study round"
assert explanatory_ax.get_ylabel() == "Prepared score (points)"
assert explanatory_ax.get_legend() is not None
assert explanatory_annotation.get_text() == "Round 5 observed separation: 7 points"
assert "caused" not in explanatory_ax.get_title().lower()
assert "do not establish a causal" in explanatory_text_alternative

supporting_readback = pd.read_csv(supporting_path)
pd.testing.assert_frame_equal(supporting_readback, supporting_data)
assert text_path.read_text(encoding="utf-8") == explanatory_text_alternative + "\n"
assert explanatory_path.is_file() and explanatory_path.stat().st_size > 1_000
explanatory_pixels = plt.imread(explanatory_path)
assert explanatory_pixels.shape[0] > 350 and explanatory_pixels.shape[1] > 700

print("Wrote:", supporting_path)
print("Wrote:", text_path)
print("Wrote:", explanatory_path)


## Human visual QA

Automated checks cannot certify the argument. Inspect the newly rendered chart and answer: does the chart match the question, audience, and claim; does each mark represent the stated prepared-summary unit; are scale, context, title, and annotation honest; are both lines recoverable without color alone; is the layout unclipped; and does the text alternative name the chart, axes, pattern, and limitation in useful language?


In [ ]:
assert supporting_path.exists() and text_path.exists() and explanatory_path.exists()
assert guided_rounds.iloc[-1]["score"] - standard_rounds.iloc[-1]["score"] == 7
print("Demo 3 fresh-run verification passed")
